# 🤖 ML Model Training Walkthrough

This notebook demonstrates the training process for all ML models in the Supply Chain SaaS platform.

## Models Covered
1. **Demand Forecasting** - Time series regression for predicting product demand
2. **Inventory Risk Classification** - Multi-class classification for stock risk levels
3. **Supplier Delay Prediction** - Binary classification for shipment delays
4. **Cost Anomaly Detection** - Unsupervised anomaly detection for cost optimization

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add backend to path
sys.path.insert(0, str(Path('../../backend').resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# Import our ML modules
from app.ml.models.demand_forecasting import DemandForecastingModel
from app.ml.models.inventory_risk_classifier import InventoryRiskClassifier
from app.ml.models.supplier_delay_predictor import SupplierDelayPredictor
from app.ml.models.cost_anomaly_detector import CostAnomalyDetector
from app.ml.training.trainer import MLTrainer

print("✅ All imports successful!")

## 2. Load Training Data

In [ ]:
DATA_DIR = Path('../../data/processed')

# Load datasets
try:
    demand_data = pd.read_csv(DATA_DIR / 'demand_data.csv')
    print(f"✅ Demand data: {demand_data.shape}")
except FileNotFoundError:
    print("⚠️ demand_data.csv not found")
    demand_data = None

try:
    classification_data = pd.read_csv(DATA_DIR / 'classification_data.csv')
    print(f"✅ Classification data: {classification_data.shape}")
except FileNotFoundError:
    print("⚠️ classification_data.csv not found")
    classification_data = None

# Preview data
if demand_data is not None:
    display(demand_data.head())
if classification_data is not None:
    display(classification_data.head())

---
## 3. Model 1: Demand Forecasting

**Objective**: Predict future product demand based on historical order data.

**Algorithms**: Linear Regression, Random Forest Regressor

**Features**: Year, Month, Day of Week, Quarter, Product ID

In [ ]:
if demand_data is not None:
    print("🚀 Training Demand Forecasting Model...")
    print("-" * 40)
    
    # Initialize model
    demand_model = DemandForecastingModel()
    
    # Train model
    results = demand_model.train(demand_data)
    
    # Display results
    print("\n📊 Training Results:")
    print(f"\nLinear Regression:")
    print(f"  RMSE: {results['linear_regression']['rmse']:.4f}")
    print(f"  MAE:  {results['linear_regression']['mae']:.4f}")
    print(f"  R²:   {results['linear_regression']['r2']:.4f}")
    
    print(f"\nRandom Forest:")
    print(f"  RMSE: {results['random_forest']['rmse']:.4f}")
    print(f"  MAE:  {results['random_forest']['mae']:.4f}")
    print(f"  R²:   {results['random_forest']['r2']:.4f}")
    
    # Save model
    model_path = Path('../../backend/app/ml/models/demand_forecasting_model.pkl')
    demand_model.save_model(str(model_path))
    print(f"\n✅ Model saved to {model_path}")
else:
    print("⚠️ Skipping - demand data not available")

### Demand Forecasting - Feature Importance

In [ ]:
if demand_data is not None and demand_model.is_trained:
    # Get feature importance from Random Forest
    feature_names = ['year', 'month', 'day_of_week', 'quarter', 'product_encoded']
    importances = demand_model.rf_model.feature_importances_
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    indices = np.argsort(importances)[::-1]
    
    plt.bar(range(len(importances)), importances[indices])
    plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=45)
    plt.title('Demand Forecasting - Feature Importance')
    plt.xlabel('Features')
    plt.ylabel('Importance')
    plt.tight_layout()
    plt.show()

---
## 4. Model 2: Inventory Risk Classification

**Objective**: Classify products into risk categories (Normal, Overstock, Stockout Risk).

**Algorithms**: Random Forest Classifier, Logistic Regression

**Features**: Stock levels, Availability, Lead times, etc.

In [ ]:
if classification_data is not None and 'inventory_risk_label' in classification_data.columns:
    print("🚀 Training Inventory Risk Classifier...")
    print("-" * 40)
    
    # Check class distribution
    print("\nClass Distribution:")
    print(classification_data['inventory_risk_label'].value_counts())
    
    # Initialize and train model
    risk_model = InventoryRiskClassifier()
    results = risk_model.train(classification_data)
    
    # Display results
    print("\n📊 Training Results:")
    print(f"\nRandom Forest Accuracy: {results['random_forest']['accuracy']:.4f}")
    print(f"Logistic Regression Accuracy: {results['logistic_regression']['accuracy']:.4f}")
    
    print("\nClassification Report (Random Forest):")
    print(results['random_forest']['classification_report'])
    
    # Save model
    model_path = Path('../../backend/app/ml/models/inventory_risk_classifier.pkl')
    risk_model.save_model(str(model_path))
    print(f"\n✅ Model saved to {model_path}")
else:
    print("⚠️ Skipping - classification data or inventory_risk_label not available")

---
## 5. Model 3: Supplier Delay Prediction

**Objective**: Predict whether a supplier shipment will be delayed.

**Algorithms**: Logistic Regression, XGBoost

In [ ]:
if classification_data is not None and 'supplier_delay_label' in classification_data.columns:
    print("🚀 Training Supplier Delay Predictor...")
    print("-" * 40)
    
    # Check class distribution
    print("\nClass Distribution:")
    print(classification_data['supplier_delay_label'].value_counts())
    
    # Initialize and train model
    delay_model = SupplierDelayPredictor()
    results = delay_model.train(classification_data)
    
    # Display results
    print("\n📊 Training Results:")
    if 'xgboost' in results:
        print(f"XGBoost Accuracy: {results['xgboost']['accuracy']:.4f}")
        if 'roc_auc' in results['xgboost']:
            print(f"XGBoost ROC-AUC: {results['xgboost']['roc_auc']:.4f}")
    if 'logistic_regression' in results:
        print(f"Logistic Regression Accuracy: {results['logistic_regression']['accuracy']:.4f}")
    
    # Save model
    model_path = Path('../../backend/app/ml/models/supplier_delay_predictor.pkl')
    delay_model.save_model(str(model_path))
    print(f"\n✅ Model saved to {model_path}")
else:
    print("⚠️ Skipping - classification data or supplier_delay_label not available")

---
## 6. Model 4: Cost Anomaly Detection

**Objective**: Detect anomalous costs that may indicate issues.

**Algorithm**: Isolation Forest (Unsupervised)

In [ ]:
if classification_data is not None:
    print("🚀 Training Cost Anomaly Detector...")
    print("-" * 40)
    
    # Initialize and train model
    anomaly_model = CostAnomalyDetector()
    results = anomaly_model.train(classification_data)
    
    # Display results
    print("\n📊 Training Results:")
    print(f"Anomaly Percentage: {results.get('anomaly_percentage', 'N/A'):.2f}%")
    print(f"Normal Count: {results.get('normal_count', 'N/A')}")
    print(f"Anomaly Count: {results.get('anomaly_count', 'N/A')}")
    
    # Save model
    model_path = Path('../../backend/app/ml/models/cost_anomaly_detector.pkl')
    anomaly_model.save_model(str(model_path))
    print(f"\n✅ Model saved to {model_path}")
else:
    print("⚠️ Skipping - classification data not available")

---
## 7. Training Summary

In [ ]:
print("\n" + "="*60)
print("📋 TRAINING SUMMARY")
print("="*60)

models_status = [
    ("Demand Forecasting", demand_data is not None),
    ("Inventory Risk Classification", classification_data is not None and 'inventory_risk_label' in (classification_data.columns if classification_data is not None else [])),
    ("Supplier Delay Prediction", classification_data is not None and 'supplier_delay_label' in (classification_data.columns if classification_data is not None else [])),
    ("Cost Anomaly Detection", classification_data is not None),
]

for model_name, was_trained in models_status:
    status = "✅ Trained" if was_trained else "⚠️ Skipped (missing data)"
    print(f"  {model_name}: {status}")

print("\n" + "="*60)
print("🎉 Training workflow complete!")
print("="*60)